# A02 · Linear Models, Softmax & Cross-Entropy

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Adib2024/aws-architect-path/blob/main/notebooks/a02_softmax_ce.ipynb)

Every classifier, including the final layer of GPT, ends in softmax and cross-entropy. Derive them from maximum likelihood, compute their gradient by hand (it's beautifully simple), and implement them in a numerically stable way.

Part of the **AI Engineer Path** (https://adib2024.github.io/aws-architect-path/#a02). In Colab, use **Runtime → Change runtime type → T4 GPU** for anything that trains.

**You will:**
- Derive cross-entropy from maximum likelihood.
- Prove that $\partial L/\partial z = p - y$ for softmax + cross-entropy.
- Implement a stable softmax using the log-sum-exp trick.
- Train a softmax classifier with manual gradient descent.

## The math

### Softmax and cross-entropy

$$p_i = \frac{e^{z_i}}{\sum_j e^{z_j}}, \qquad L = -\log p_y$$

Worked with three classes, true class = 0.

**Worked example**

1. $z = (2.0,\ 1.0,\ 0.1)$ (logits)
2. $e^{z} = (7.389,\ 2.718,\ 1.105),\ \sum = 11.212$
3. $p = (0.659,\ 0.242,\ 0.099)$ (sums to 1)
4. $L = -\ln 0.659 = 0.417$ (the loss for this example)

### The gradient is p − y

$$\frac{\partial L}{\partial z_k} = p_k - y_k$$

From $L = -z_y + \log\sum_j e^{z_j}$: the derivative of the first term is $-\mathbb{1}[k=y]$, of the second is $e^{z_k}/\sum_j e^{z_j} = p_k$.

**Worked example**

1. $y = (1, 0, 0)$ (one-hot label)
2. $\nabla_z L = (0.659-1,\ 0.242,\ 0.099) = (-0.341,\ 0.242,\ 0.099)$ (gradient descent raises z₀ and lowers the others)
3. $\nabla_W L = (p - y)\,x^{\top}$ (chain rule through z = Wx + b)

### Log-sum-exp stability

$$\log\sum_j e^{z_j} = m + \log\sum_j e^{z_j - m}, \quad m = \max_j z_j$$

$e^{1000}$ overflows float32 (max ≈ $3.4\times10^{38}$, so anything above $z \approx 88.7$ is inf). Subtracting the max makes the largest exponent $e^0 = 1$.

**Worked example**

1. $z = (1000, 1001, 1002) \Rightarrow z - m = (-2, -1, 0)$
2. $p = (0.090,\ 0.245,\ 0.665)$ (same as softmax(0, 1, 2))

### Binary case (one neuron)

$$\sigma(z) = \frac{1}{1+e^{-z}}, \quad \frac{\partial L}{\partial z} = \sigma(z) - y$$

$x = (1, 2),\ w = (0.8, -0.2),\ b = 0.1,\ y = 1$.

**Worked example**

1. $z = 0.8 - 0.4 + 0.1 = 0.5,\ \ \sigma = 0.6225$
2. $L = -\ln 0.6225 = 0.474$
3. $\partial L/\partial w = (\sigma - y)\,x = -0.3775 \cdot (1, 2) = (-0.378,\ -0.755)$

## Lab: Softmax, cross-entropy and gradients from scratch

Reproduce every number in the worked examples, then train a 3-class classifier with your own update rule.

### Step 1

Reproduce the worked example, then check against PyTorch's fused version.

In [ ]:
import torch, torch.nn.functional as F
z = torch.tensor([2.0, 1.0, 0.1])
p = torch.exp(z) / torch.exp(z).sum()
print(p)                                   # tensor([0.6590, 0.2424, 0.0986])
print(-torch.log(p[0]))                    # tensor(0.4170)
print(F.cross_entropy(z.unsqueeze(0), torch.tensor([0])))   # same

### Step 2

Verify the gradient $p - y$ with autograd.

In [ ]:
z = torch.tensor([[2.0, 1.0, 0.1]], requires_grad=True)
F.cross_entropy(z, torch.tensor([0])).backward()
print(z.grad)                              # tensor([[-0.3410,  0.2424,  0.0986]])

### Step 3

See the overflow and fix it with log-sum-exp.

In [ ]:
big = torch.tensor([1000., 1001., 1002.])
print(torch.exp(big) / torch.exp(big).sum())   # tensor([nan, nan, nan])
def stable_softmax(z):
    z = z - z.max()
    e = z.exp()
    return e / e.sum()
print(stable_softmax(big))                     # tensor([0.0900, 0.2447, 0.6652])
print(torch.logsumexp(big, 0))                 # 1002.4076

### Step 4

Train softmax regression on three Gaussian blobs with a hand-written gradient step.

In [ ]:
torch.manual_seed(0)
N = 300
centers = torch.tensor([[0., 0.], [3., 3.], [0., 4.]])
y = torch.randint(0, 3, (N,))
X = centers[y] + torch.randn(N, 2)
W = torch.zeros(2, 3, requires_grad=True)
b = torch.zeros(3, requires_grad=True)
for step in range(301):
    logits = X @ W + b
    loss = F.cross_entropy(logits, y)
    loss.backward()
    with torch.no_grad():
        W -= 0.1 * W.grad
        b -= 0.1 * b.grad
        W.grad.zero_(); b.grad.zero_()
    if step % 100 == 0:
        acc = (logits.argmax(1) == y).float().mean().item()
        print(f"step {step:3d}  loss {loss.item():.3f}  acc {acc:.2%}")
# step 0 loss is ln(3) = 1.099: uniform predictions, a sanity check you should always do

### Clean up

Nothing to clean up.

## Check yourself

**1. Logits $(3, 3, 3)$. What is the cross-entropy loss for any true class?**

A. $0$  
B. $\ln 3 \approx 1.099$  
C. $3$  
D. $1/3$

<details><summary>Answer</summary>

**B.** Equal logits give $p = 1/3$ each, so $L = -\ln(1/3) = \ln 3$.

</details>

**2. You add 100 to every logit. What happens to softmax?**

A. Probabilities shift towards the largest logit  
B. Nothing: softmax is shift-invariant  
C. It overflows and returns NaN in PyTorch  
D. All probabilities become equal

<details><summary>Answer</summary>

**B.** $e^{z_i + c}/\sum e^{z_j + c} = e^{z_i}/\sum e^{z_j}$. PyTorch's softmax subtracts the max internally, so no overflow.

</details>

**3. With $p = (0.7, 0.2, 0.1)$ and true class 2, what is $\partial L/\partial z$?**

A. $(0.7, 0.2, -0.9)$  
B. $(-0.3, 0.2, 0.1)$  
C. $(0.7, 0.2, 0.1)$  
D. $(0, 0, -0.9)$

<details><summary>Answer</summary>

**A.** $p - y = (0.7, 0.2, 0.1) - (0, 0, 1) = (0.7, 0.2, -0.9)$.

</details>